# 🏥 Tech Challenge FIAP - Fase 1
## Notebook 04: Avaliação Detalhada dos Modelos

---

### 📋 **Objetivos deste Notebook**

1. ✅ Carregar modelos treinados
2. ✅ Matriz de Confusão para cada modelo
3. ✅ Curvas ROC e AUC
4. ✅ Curvas Precision-Recall
5. ✅ Análise de erros (falsos positivos/negativos)
6. ✅ Comparação estatística
7. ✅ Relatório final de métricas

---

**Input**: Modelos treinados + Predições
**Output**: Análise completa de performance + Visualizações + Relatório

## 📚 1. Importação de Bibliotecas

In [ ]:
# Manipulação de dados
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Métricas e avaliação
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
    matthews_corrcoef,
    cohen_kappa_score
)

# Carregar modelos
import joblib
import pickle

# Configurações
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✅ Bibliotecas importadas com sucesso!")

## 📂 2. Carregamento de Dados e Modelos

In [ ]:
# Carregar dados de teste
print("📥 Carregando dados...")
X_test = np.load('../data/processed/X_test.npy')
y_test = np.load('../data/processed/y_test.npy')

print(f"✅ Dados carregados: {X_test.shape}")

In [ ]:
# Carregar predições
with open('../models/predictions.pkl', 'rb') as f:
    predictions_dict = pickle.load(f)

predictions = predictions_dict['predictions']
predictions_proba = predictions_dict['predictions_proba']

print("✅ Predições carregadas")
print(f"   Modelos: {list(predictions.keys())}")

In [ ]:
# Carregar metadados
with open('../models/model_metadata.pkl', 'rb') as f:
    metadata = pickle.load(f)

model_names = metadata['models']
print(f"✅ Modelos a avaliar: {model_names}")

## 📊 3. Matrizes de Confusão

In [ ]:
# Função para plotar matriz de confusão bonita
def plot_confusion_matrix(y_true, y_pred, model_name, ax):
    """
    Plota matriz de confusão formatada
    """
    cm = confusion_matrix(y_true, y_pred)
    
    # Calcular percentuais
    cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    
    # Plot
    sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', ax=ax, 
                cbar=False, square=True, linewidths=2)
    
    # Adicionar valores e percentuais
    for i in range(2):
        for j in range(2):
            text = f'{cm[i, j]}\n({cm_percent[i, j]:.1f}%)'
            ax.text(j + 0.5, i + 0.5, text, ha='center', va='center',
                   fontsize=11, fontweight='bold',
                   color='white' if cm[i, j] > cm.max()/2 else 'black')
    
    ax.set_xlabel('Predição', fontweight='bold')
    ax.set_ylabel('Real', fontweight='bold')
    ax.set_title(f'{model_name}', fontweight='bold', pad=10)
    ax.set_xticklabels(['Maligno (0)', 'Benigno (1)'])
    ax.set_yticklabels(['Maligno (0)', 'Benigno (1)'])

print("✅ Função de plotagem definida")

In [ ]:
# Plotar todas as matrizes de confusão
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()

for idx, model_name in enumerate(model_names):
    y_pred = predictions[model_name]
    plot_confusion_matrix(y_test, y_pred, model_name, axes[idx])

plt.suptitle('Matrizes de Confusão - Todos os Modelos', 
             fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('../reports/figures/12_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Gráfico salvo em: reports/figures/12_confusion_matrices.png")

## 📈 4. Análise Detalhada de Erros

In [ ]:
# Análise de erros para cada modelo
print("🔍 ANÁLISE DE ERROS")
print("="*100)

error_analysis = []

for model_name in model_names:
    y_pred = predictions[model_name]
    cm = confusion_matrix(y_test, y_pred)
    
    # Extrair componentes
    tn, fp, fn, tp = cm.ravel()
    
    # Calcular taxas
    total = tn + fp + fn + tp
    
    error_analysis.append({
        'Model': model_name,
        'True Negatives': tn,
        'False Positives': fp,
        'False Negatives': fn,
        'True Positives': tp,
        'Total Correct': tn + tp,
        'Total Errors': fp + fn,
        'Error Rate (%)': (fp + fn) / total * 100
    })
    
    print(f"\n🤖 {model_name}:")
    print(f"   ✅ Verdadeiros Negativos (TN): {tn}")
    print(f"   ✅ Verdadeiros Positivos (TP): {tp}")
    print(f"   ❌ Falsos Positivos (FP): {fp} - Alarmes falsos")
    print(f"   ❌ Falsos Negativos (FN): {fn} - CRÍTICO! Câncer não detectado")
    print(f"   📊 Taxa de Erro: {(fp + fn) / total * 100:.2f}%")

print("\n" + "="*100)

In [ ]:
# DataFrame com análise de erros
error_df = pd.DataFrame(error_analysis)

print("📊 TABELA DE ERROS:")
print("="*100)
print(error_df.to_string(index=False))
print("="*100)

# Salvar
error_df.to_csv('../reports/error_analysis.csv', index=False)
print("\n✅ Análise de erros salva em: reports/error_analysis.csv")

In [ ]:
# Visualização de erros
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Falsos Positivos vs Falsos Negativos
x = np.arange(len(error_df))
width = 0.35

axes[0].bar(x - width/2, error_df['False Positives'], width, 
           label='Falsos Positivos', color='#FFA07A', alpha=0.8)
axes[0].bar(x + width/2, error_df['False Negatives'], width, 
           label='Falsos Negativos', color='#FF6B6B', alpha=0.8)

axes[0].set_xlabel('Modelo', fontweight='bold')
axes[0].set_ylabel('Quantidade', fontweight='bold')
axes[0].set_title('Análise de Erros por Tipo', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(error_df['Model'], rotation=15, ha='right')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Gráfico 2: Taxa de Erro
colors = ['#4ECDC4' if err < 5 else '#FFA07A' if err < 10 else '#FF6B6B' 
          for err in error_df['Error Rate (%)']]

bars = axes[1].barh(error_df['Model'], error_df['Error Rate (%)'], 
                    color=colors, alpha=0.8)
axes[1].set_xlabel('Taxa de Erro (%)', fontweight='bold')
axes[1].set_title('Taxa de Erro por Modelo', fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

# Adicionar valores
for i, bar in enumerate(bars):
    width_val = bar.get_width()
    axes[1].text(width_val, bar.get_y() + bar.get_height()/2, 
                f'{width_val:.2f}%', ha='left', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/13_error_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Gráfico salvo em: reports/figures/13_error_analysis.png")

## 📈 5. Curvas ROC (Receiver Operating Characteristic)

In [ ]:
# Plotar curvas ROC para todos os modelos
plt.figure(figsize=(10, 8))

colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']

for idx, model_name in enumerate(model_names):
    y_pred_proba = predictions_proba[model_name]
    
    # Calcular curva ROC
    fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    # Plotar
    plt.plot(fpr, tpr, color=colors[idx], lw=2.5, 
             label=f'{model_name} (AUC = {roc_auc:.4f})', alpha=0.8)

# Linha diagonal (classificador aleatório)
plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Classificador Aleatório (AUC = 0.5000)')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Taxa de Falsos Positivos (FPR)', fontsize=12, fontweight='bold')
plt.ylabel('Taxa de Verdadeiros Positivos (TPR)', fontsize=12, fontweight='bold')
plt.title('Curvas ROC - Comparação de Modelos', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/figures/14_roc_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Gráfico salvo em: reports/figures/14_roc_curves.png")
print("\n💡 AUC próximo de 1.0 = Excelente")
print("   AUC = 0.5 = Aleatório")

## 📊 6. Curvas Precision-Recall

In [ ]:
# Plotar curvas Precision-Recall
plt.figure(figsize=(10, 8))

for idx, model_name in enumerate(model_names):
    y_pred_proba = predictions_proba[model_name]
    
    # Calcular curva Precision-Recall
    precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba)
    avg_precision = average_precision_score(y_test, y_pred_proba)
    
    # Plotar
    plt.plot(recall, precision, color=colors[idx], lw=2.5,
             label=f'{model_name} (AP = {avg_precision:.4f})', alpha=0.8)

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Recall (Sensibilidade)', fontsize=12, fontweight='bold')
plt.ylabel('Precision (Precisão)', fontsize=12, fontweight='bold')
plt.title('Curvas Precision-Recall - Comparação de Modelos', fontsize=14, fontweight='bold')
plt.legend(loc='lower left', fontsize=10)
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/figures/15_precision_recall_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Gráfico salvo em: reports/figures/15_precision_recall_curves.png")
print("\n💡 Average Precision (AP) resume a curva em um único número")
print("   AP alto = Bom balanço entre precisão e recall")

## 📋 7. Relatório de Classificação Completo

In [ ]:
# Gerar relatório de classificação para cada modelo
print("📋 RELATÓRIOS DE CLASSIFICAÇÃO DETALHADOS")
print("="*100)

for model_name in model_names:
    y_pred = predictions[model_name]
    
    print(f"\n🤖 {model_name}")
    print("-" * 80)
    print(classification_report(y_test, y_pred, 
                                target_names=['Maligno (0)', 'Benigno (1)'],
                                digits=4))
    print("-" * 80)

## 📊 8. Métricas Adicionais

In [ ]:
# Calcular métricas adicionais
additional_metrics = []

print("📊 MÉTRICAS ADICIONAIS")
print("="*100)

for model_name in model_names:
    y_pred = predictions[model_name]
    y_pred_proba = predictions_proba[model_name]
    
    # Matthews Correlation Coefficient
    mcc = matthews_corrcoef(y_test, y_pred)
    
    # Cohen's Kappa
    kappa = cohen_kappa_score(y_test, y_pred)
    
    # Specificity (Taxa de Verdadeiros Negativos)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp)
    
    # Sensitivity (Recall)
    sensitivity = recall_score(y_test, y_pred)
    
    additional_metrics.append({
        'Model': model_name,
        'MCC': mcc,
        "Cohen's Kappa": kappa,
        'Specificity': specificity,
        'Sensitivity (Recall)': sensitivity
    })
    
    print(f"\n🤖 {model_name}:")
    print(f"   Matthews Correlation Coef: {mcc:.4f}")
    print(f"   Cohen's Kappa: {kappa:.4f}")
    print(f"   Specificity (TNR): {specificity:.4f}")
    print(f"   Sensitivity (TPR): {sensitivity:.4f}")

print("\n" + "="*100)

In [ ]:
# DataFrame com métricas adicionais
additional_df = pd.DataFrame(additional_metrics)
additional_df = additional_df.round(4)

print("📊 TABELA DE MÉTRICAS ADICIONAIS:")
print("="*100)
print(additional_df.to_string(index=False))
print("="*100)

# Salvar
additional_df.to_csv('../reports/additional_metrics.csv', index=False)
print("\n✅ Métricas adicionais salvas em: reports/additional_metrics.csv")

## 🏆 9. Ranking Final dos Modelos

In [ ]:
# Carregar resultados do Notebook 03
results_df = pd.read_csv('../reports/model_comparison.csv')

# Combinar com métricas adicionais
final_results = pd.merge(results_df, additional_df, on='Model')

print("📊 TABELA COMPLETA DE RESULTADOS")
print("="*120)
print(final_results.to_string(index=False))
print("="*120)

# Salvar resultados finais
final_results.to_csv('../reports/final_results.csv', index=False)
print("\n✅ Resultados finais salvos em: reports/final_results.csv")

In [ ]:
# Criar ranking por métrica
print("\n🏆 RANKING POR MÉTRICA")
print("="*100)

metrics_to_rank = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'MCC']

for metric in metrics_to_rank:
    ranked = final_results.sort_values(metric, ascending=False)[['Model', metric]]
    print(f"\n📊 {metric}:")
    for idx, row in ranked.iterrows():
        medal = ['🥇', '🥈', '🥉', '4️⃣'][idx] if idx < 4 else ''
        print(f"   {medal} {row['Model']}: {row[metric]:.4f}")

print("\n" + "="*100)

## 🎯 10. Recomendações Finais

In [ ]:
# Identificar melhor modelo geral
# Vamos usar uma média ponderada das métricas principais

weights = {
    'Accuracy': 0.15,
    'Precision': 0.15,
    'Recall': 0.25,      # Maior peso (crítico em medicina)
    'F1-Score': 0.20,
    'ROC-AUC': 0.15,
    'MCC': 0.10
}

final_results['Weighted Score'] = sum(
    final_results[metric] * weight 
    for metric, weight in weights.items()
)

# Ordenar por score ponderado
final_results_sorted = final_results.sort_values('Weighted Score', ascending=False)

print("🎯 RECOMENDAÇÃO FINAL (Score Ponderado)")
print("="*100)
print("\nPesos utilizados:")
for metric, weight in weights.items():
    print(f"   {metric}: {weight*100:.0f}%")

print("\n🏆 RANKING FINAL:")
for idx, row in final_results_sorted.iterrows():
    medal = ['🥇', '🥈', '🥉', '4️⃣'][idx] if idx < 4 else ''
    print(f"   {medal} {row['Model']}: Score = {row['Weighted Score']:.4f}")

best_model = final_results_sorted.iloc[0]['Model']
print(f"\n✅ MODELO RECOMENDADO: {best_model}")
print("="*100)

In [ ]:
# Salvar ranking final
final_results_sorted.to_csv('../reports/final_ranking.csv', index=False)
print("✅ Ranking final salvo em: reports/final_ranking.csv")

## 📊 11. Resumo da Avaliação

In [ ]:
print("="*100)
print("📊 RESUMO DA AVALIAÇÃO")
print("="*100)

print("\n✅ ANÁLISES REALIZADAS:")
print("   1. ✅ Matrizes de Confusão (4 modelos)")
print("   2. ✅ Análise de Erros (FP, FN)")
print("   3. ✅ Curvas ROC")
print("   4. ✅ Curvas Precision-Recall")
print("   5. ✅ Relatórios de Classificação")
print("   6. ✅ Métricas Adicionais (MCC, Kappa, etc.)")
print("   7. ✅ Ranking Final")

print("\n📊 MÉTRICAS CALCULADAS:")
metrics_list = list(final_results.columns[1:])
for i, metric in enumerate(metrics_list, 1):
    print(f"   {i}. {metric}")

print(f"\n🏆 MELHOR MODELO: {best_model}")

print("\n💾 ARQUIVOS GERADOS:")
print("   • reports/model_comparison.csv")
print("   • reports/error_analysis.csv")
print("   • reports/additional_metrics.csv")
print("   • reports/final_results.csv")
print("   • reports/final_ranking.csv")
print("   • reports/figures/12_confusion_matrices.png")
print("   • reports/figures/13_error_analysis.png")
print("   • reports/figures/14_roc_curves.png")
print("   • reports/figures/15_precision_recall_curves.png")

print("\n🎯 PONTOS IMPORTANTES:")
print("   • Todos os modelos têm performance excelente (>95% accuracy)")
print("   • Recall é crítico em diagnóstico médico (minimizar FN)")
print("   • ROC-AUC próximo de 1.0 indica excelente discriminação")
print("   • Dataset é bem comportado e separável")

print("\n" + "="*100)

## 📝 12. Observações Importantes

### **🩺 Contexto Médico:**

1. **Falsos Negativos (FN) são CRÍTICOS**:
   - Paciente com câncer é classificado como saudável
   - Pode atrasar tratamento e comprometer prognóstico
   - **Recall (Sensibilidade)** deve ser maximizado

2. **Falsos Positivos (FP) são INDESEJÁVEIS mas menos graves**:
   - Paciente saudável é classificado como tendo câncer
   - Gera ansiedade e exames adicionais
   - Mas permite detecção precoce com mais investigação

3. **Trade-off Precision vs Recall**:
   - Em medicina: Recall > Precision
   - Melhor "alarme falso" do que "não detectar"
   - Exames confirmatórios filtram falsos positivos

### **📊 Interpretação das Métricas:**

- **Accuracy**: Performance geral (não é suficiente sozinha!)
- **Precision**: Quando prediz positivo, está correto?
- **Recall**: De todos os positivos reais, quantos acertou?
- **F1-Score**: Média harmônica de Precision e Recall
- **ROC-AUC**: Capacidade de discriminação (0.5 = aleatório, 1.0 = perfeito)
- **MCC**: Correlação entre predição e realidade (-1 a 1)
- **Cohen's Kappa**: Concordância além do acaso

### **⚠️ IMPORTANTE:**

> **Estes modelos são ferramentas de APOIO à decisão médica.**  
> O diagnóstico final SEMPRE deve ser confirmado por:  
> 1. Profissional médico qualificado  
> 2. Exames complementares (biópsia, histopatologia)  
> 3. Avaliação clínica completa do paciente  

---

## ➡️ Próximos Passos

### **No Notebook 05 - Explicabilidade (SHAP):**

1. **SHAP Values** (OBRIGATÓRIO FIAP):
   - Explicar predições individuais
   - Identificar features mais importantes
   - Validar com conhecimento médico

2. **Feature Importance**:
   - Ranking de importância
   - Comparação entre modelos
   - Interpretação clínica

3. **Casos de Uso**:
   - Exemplos práticos
   - Análise de casos específicos
   - Discussão de limitações

---

## ✅ Conclusão do Notebook 04

**Status**: ✅ Completo

**Análises Realizadas**: 7

**Gráficos Gerados**: 4

**Próximo**: `05_explicabilidade.ipynb` (SHAP - OBRIGATÓRIO)